In [1]:
import glob
import os
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe

# ── Dask ─────────────────────────────────────────────────────────────────────
import dask
import dask.bag as db
from dask.distributed import Client, LocalCluster, progress, wait

from swot_analysis import compute_pass_spectra
from swot_analysis import load_swot_l2_expert, load_swot_l2_unsmoothed 

In [2]:
# ── Input files ───────────────────────────────────────────────────────────────
DATA_DIR    = '/Volumes/PortableSSD/'#'/Users/zoecas/Documents/data/'
#DATA_DIR    = '/Users/zoecas/Documents/data/'

# ── One sub-directory per month of SWOT data ────────────────────────────────
# Every file found under each entry is tagged with the (year, month) inferred
# from that sub-directory's name (see parse_year_month below).
DATA_SUBDIRS = ['SWOT_L2_LR_SSH_Expert_0124/',
                'SWOT_L2_LR_SSH_Expert_0224/',
                'SWOT_L2_LR_SSH_Expert_0324/',
               'SWOT_L2_LR_SSH_Expert_0424/',
               'SWOT_L2_LR_SSH_Expert_0524/',
               'SWOT_L2_LR_SSH_Expert_0624/',
               'SWOT_L2_LR_SSH_Expert_0724/',
               'SWOT_L2_LR_SSH_Expert_0824/',
               'SWOT_L2_LR_SSH_Expert_0924/',
               'SWOT_L2_LR_SSH_Expert_1024/',
               'SWOT_L2_LR_SSH_Expert_1124/',
               'SWOT_L2_LR_SSH_Expert_1224/',
               ]
FILE_PATTERN = 'SWOT_L2_LR_SSH_Expert_*.nc'

import re

def parse_year_month(subdir_name, overrides=None):
    """
    Infer (year, month) from a sub-directory name.

    Default rule: the trailing 4 digits are read as MMYY, e.g.
    'SWOT_L2_0225' -> month=2, year=2025.

    For directories that don't follow this convention, pass a dict via
    `overrides` mapping subdir name -> (year, month); it takes precedence
    over the automatic parsing.
    """
    if overrides and subdir_name in overrides:
        return overrides[subdir_name]
    name = subdir_name.rstrip('/').rstrip('\\')
    match = re.search(r'(\d{2})(\d{2})$', name)
    if not match:
        raise ValueError(
            f"Could not infer year/month from subdirectory name '{subdir_name}'. "
            f"Add an explicit entry to DATA_SUBDIR_OVERRIDES instead."
        )
    mm, yy = match.groups()
    month = int(mm)
    year = 2000 + int(yy)
    if not (1 <= month <= 12):
        raise ValueError(f"Parsed invalid month={month} from '{subdir_name}'.")
    return year, month

# Manual overrides for sub-directories that don't follow the MMYY naming
# convention above, e.g. {'my_weird_folder_name': (2025, 7)}
DATA_SUBDIR_OVERRIDES = {}

# ── Region of interest ────────────────────────────────────────────────────────
LAT_MIN, LAT_MAX = -60.0,60.0
LON_MIN, LON_MAX = 0.0,360.0

# ── Spectral parameters ───────────────────────────────────────────────────────
SEGMENT_LENGTH_KM      = 1000.0
OVERLAP                = 0.5
MAX_GAP_FRACTION       = 0.25
DETREND                = "constant"
SWH_VAR                = False
CROSS_TRACK_INTEGRATED = True
MAX_NAN_FRACTION_2D = 0.35   # start here given n_pixels_used ~25-27 of ~34; tune after checking n_segments_used_2d

# ── Box size ──────────────────────────────────────────────────────────────────
BOX_DEG = 2.0

# ── Expert-product variable names ─────────────────────────────────────────────
SSH_VAR = 'ssha_karin_2'
HRET    = True   # add internal tide correction (height_cor_xover + internal_tide_hret)

# ── Dask cluster parameters ───────────────────────────────────────────────────
N_WORKERS          = 8       # number of worker processes
THREADS_PER_WORKER = 1       # do NOT increase, core functions contain lops
MEMORY_LIMIT       = '2GB'   # per worker RAM cap

In [5]:
filepath = os.listdir(Path(DATA_DIR+DATA_SUBDIRS[0]))[0]

In [8]:
filepath

'SWOT_L2_LR_SSH_Expert_010_147_20240130T053047_20240130T062216_PGD0_02.nc'

In [9]:
data_expert = xr.open_dataset(DATA_DIR+DATA_SUBDIRS[0]+filepath)

In [10]:
data_expert

<xarray.Dataset> Size: 400MB
Dimensions:                                (num_lines: 9866, num_pixels: 69,
                                            num_sides: 2)
Coordinates:
    latitude_nadir                         (num_lines) float64 79kB ...
    longitude_nadir                        (num_lines) float64 79kB ...
    latitude                               (num_lines, num_pixels) float64 5MB ...
    longitude                              (num_lines, num_pixels) float64 5MB ...
Dimensions without coordinates: num_lines, num_pixels, num_sides
Data variables: (12/99)
    time                                   (num_lines) datetime64[ns] 79kB ...
    time_tai                               (num_lines) datetime64[ns] 79kB ...
    ssh_karin                              (num_lines, num_pixels) float64 5MB ...
    ssh_karin_qual                         (num_lines, num_pixels) float64 5MB ...
    ssh_karin_uncert                       (num_lines, num_pixels) float64 5MB ...
    ssha_karin                             (num_lines, num_pixels) float64 5MB ...
    ...                                     ...
    swh_ssb_cor_source                     (num_lines, num_pixels) float32 3MB ...
    swh_ssb_cor_source_2                   (num_lines, num_pixels) float32 3MB ...
    wind_speed_ssb_cor_source              (num_lines, num_pixels) float32 3MB ...
    wind_speed_ssb_cor_source_2            (num_lines, num_pixels) float32 3MB ...
    volumetric_correlation                 (num_lines, num_pixels) float64 5MB ...
    volumetric_correlation_uncert          (num_lines, num_pixels) float64 5MB ...
Attributes: (12/62)
    Conventions:                                   CF-1.7
    title:                                         Level 2 Low Rate Sea Surfa...
    institution:                                   CNES
    source:                                        Ka-band radar interferometer
    history:                                       2025-09-01T03:01:42Z : Cre...
    platform:                                      SWOT
    ...                                            ...
    ellipsoid_semi_major_axis:                     6378137.0
    ellipsoid_flattening:                          0.0033528106647474805
    good_ocean_data_percent:                       65.6338968674003
    ssha_variance:                                 3.06478869065094
    references:                                    V1.4.1
    equator_longitude:                             -89.83

In [12]:
xr.open_dataarray('/Users/zoecas/Downloads/SWOT_GPS_2PsP026_229_20241231_235043_20250101_004210.nc')

ValueError: Given file dataset contains no data variables.